# SynnoDB: From DuckDB to Bespoke in One Import

SynnoDB is a drop-in replacement for DuckDB that transparently accelerates your SQL queries
with auto-generated bespoke C++ engines - while falling back to DuckDB for everything else.
No schema changes. No query rewrites. One import.

This notebook walks through the full journey for **TPC-H Q1-Q5**:

1. **Baseline** - run Q1-Q5 on vanilla DuckDB at SF20, 10 parameter instantiations each.
2. **Generate** - point SynnoDB at a `queries.json` file and let it write the engine.
3. **Drop in** - replace one import, re-run the identical queries, compare the numbers.

It is **self-contained**: point it at a fresh (non-existent) data root and it generates the
TPC-H parquet itself before running anything - no manual download or external tooling.

### Prerequisites

**Install SynnoDB** - it lives on [PyPI](https://pypi.org/project/synnodb/), so a single
`pip install` pulls in every Python dependency. This demo *generates* an engine, so add the
`factory` extra:

```bash
pip install "synnodb[factory]"
```

(Working from a source checkout instead? `uv sync --extra factory` builds the same environment -
see the repo's Development section.)

**System libraries** - a C++ toolchain and the Arrow/Parquet dev headers the generated engine is
compiled against; `cloc` (optional) lets the factory report generated-code size. On Debian/Ubuntu:

```bash
sudo apt install -y build-essential cloc                # C++ compiler (+ optional cloc)

# Apache Arrow + Parquet development libraries
wget https://packages.apache.org/artifactory/arrow/$(lsb_release --id --short | tr 'A-Z' 'a-z')/apache-arrow-apt-source-latest-$(lsb_release --codename --short).deb
sudo apt install -y -V ./apache-arrow-apt-source-latest-$(lsb_release --codename --short).deb
sudo apt update
sudo apt install -y libarrow-dev libparquet-dev parquet-tools
```

**Model access** - the factory calls `MODEL` over an OpenAI-compatible API. Put the provider key
in a repo-root `.env` (loaded below): `ANTHROPIC_API_KEY=...` for the default
`anthropic/claude-sonnet-5`, or `OPENROUTER_API_KEY=...` for an `openrouter/...` model. For a
self-hosted model, point at its endpoint with `LLM_API_BASE=http://your-host:PORT/v1` and set
`OPENAI_API_KEY` to any non-empty placeholder.

## Setup

One knob - the **data root**. Set `SYNNO_DATA_DIR` (env or `.env`) to where your TPC-H data
should live; unset, it defaults to a project-local `.synno_data/`. It does **not** need to
exist yet: the next cell generates the TPC-H parquet into it on first run.

In [9]:
import os, json, time, statistics
from pathlib import Path

from dotenv import load_dotenv

from synnodb.utils.path_utils import repo_root
load_dotenv()  # let SYNNO_DATA_DIR / SYNNO_ENGINES_DIR / SYNNO_WORKSPACE come from a .env

# The data root holds everything: parquet, engines, workspace. Honor SYNNO_DATA_DIR if set,
# else default to a project-local .synno_data/. It need not exist yet - the next cell
# materializes the TPC-H parquet into it.
DATA_ROOT = Path(os.environ.get("SYNNO_DATA_DIR") or repo_root() / ".synno_data")
PARQUET_DIR = DATA_ROOT / "workloads" / "tpch" / "tpch_parquet"
ENGINES_DIR = DATA_ROOT / "engines"

SF = 5
SCALE_FACTORS = (1, 2, SF)        # sf1/sf2: cheap correctness rungs; sf20: benchmark + serving
SF_DIR = PARQUET_DIR / f"sf{SF}"  # the benchmark scale factor's parquet

TABLES = [
    "customer",
    "lineitem",
    "nation",
    "orders",
    "part",
    "partsupp",
    "region",
    "supplier",
]

MODEL = os.environ.get(
    "SYNNO_MODEL", "anthropic/claude-sonnet-5"
)  # e.g. "anthropic/claude-sonnet-4-6", "gpt-5.4", "openrouter/z-ai/glm-5.2"
MODEL_EXTRA_BODY = json.loads(os.environ.get("SYNNO_MODEL_EXTRA_BODY", "null"))

print("Data root   :", DATA_ROOT)
print("Parquet dir :", PARQUET_DIR)
print("Engines dir :", ENGINES_DIR)
print("Model       :", MODEL)

Data root   : /home/jwehrstein/SynnoDB/.synno_data
Parquet dir : /home/jwehrstein/SynnoDB/.synno_data/workloads/tpch/tpch_parquet
Engines dir : /home/jwehrstein/SynnoDB/.synno_data/engines
Model       : anthropic/claude-sonnet-5


### Generate the TPC-H parquet (first run only)

To keep the notebook self-contained we materialize the TPC-H tables ourselves with DuckDB's
built-in `dbgen` - no external download, no `dbgen` binary. Parquet is written where the
framework looks for a built-in-style workload:

```
<DATA_ROOT>/workloads/tpch/tpch_parquet/sf<N>/<table>.parquet
```

We generate **sf1** and **sf2** (the cheap rungs the engine build validates correctness
against) and **sf20** (the benchmark / serving scale). The step is idempotent - tables already
on disk are skipped - and a one-time cost. **sf20 is ~15-20 GB and takes a while**; make sure
you have the disk. Generation caps DuckDB's memory and spills to a temp directory so it does
not OOM at the larger scale factor.

In [10]:
from synnodb.workloads.dataset.gen_tpc_h_data import ensure_tpch_parquet

ensure_tpch_parquet(PARQUET_DIR, SCALE_FACTORS, TABLES)

sf1: present (8 tables), skipping
sf2: present (8 tables), skipping
sf5: present (8 tables), skipping


---
## Step 1 - Workload Registration

We run **Q1-Q5 on vanilla DuckDB** at SF20: 10 instantiations per query
(different placeholder values, drawn from the actual data), total wall-clock time recorded.
These exact SQL strings will be reused in Step 3 so the comparison is apples-to-apples.

### Register the BYO workload

The workload is described by a single self-describing JSON file. Each entry carries its SQL
template **and** a typed **spec** for each `[PLACEHOLDER]` slot - declaring its value *space*,
which is sampled at run time. A scalar placeholder is an `int`/`float`/`date`/`categorical`
spec; correlated or distinct placeholders share a `param_groups` spec:

```json
"6": {
  "sql": "... l_discount between [DISCOUNT] - 0.01 ... l_quantity < [QUANTITY] ...",
  "params": {
    "DATE":     { "type": "date",  "min": "1993-01-01", "max": "1997-01-01" },
    "DISCOUNT": { "type": "float", "min": 0.02, "max": 0.09, "step": 0.01 },
    "QUANTITY": { "type": "int",   "min": 24, "max": 25 }
  }
},
"7": {
  "sql": "... n1.n_name = '[NATION1]' ... n2.n_name = '[NATION2]' ...",
  "param_groups": [
    { "type": "sample", "placeholders": ["NATION1", "NATION2"], "domain": ["GERMANY", "CHINA", ...], "distinct": true }
  ]
}
```

`register_workload_from_json` reads it and derives the schema from the parquet via DuckDB.
Each placeholder's spec is sampled with the run's seeded RNG (a range → a uniform draw, a
`categorical` → a choice, a group → one joint row), so correlated placeholders stay aligned.
The typed spec is exactly what a BI dashboard renders as a slider (`int`/`float`), a dropdown
(`categorical`), or a date-picker (`date`).

In [11]:
import random
from synnodb.workloads.byo_workload import register_workload_from_json

QUERIES_JSON = Path("queries.json")  # lives next to this notebook

spec = register_workload_from_json(
    name="tpch_byo",
    queries_json=QUERIES_JSON,
    parquet_dir=PARQUET_DIR,
    scale_factors=SCALE_FACTORS,
    schema_example_table="lineitem",
)

print("Workload :", spec.name)
print("Tables   :", spec.tables)
print("Queries  :", spec.all_query_ids)

2026-07-03 10:58:30 INFO:synnodb.workloads.byo_workload:Query input: detected key style ['<bare>']; using bare ids ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22']
Workload : tpch_byo
Tables   : ('customer', 'lineitem', 'nation', 'orders', 'part', 'partsupp', 'region', 'supplier')
Queries  : ('1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22')


Here is what the queries look like - SQL templates with `[PLACEHOLDER]` slots, plus the typed
specs that define each slot's value space:

In [12]:
queries = json.loads(QUERIES_JSON.read_text())
for qid, entry in queries.items():
    print(f"=== Q{qid} ===")
    print(entry["sql"][:240], "...")
    print("params      :", entry.get("params", {}))
    print("param_groups:", entry.get("param_groups", []))
    print()

=== Q1 ===
select 
    l_returnflag,  
    l_linestatus,  
    sum(l_quantity) as sum_qty, 
    sum(l_extendedprice) as sum_base_price, 
    sum(l_extendedprice*(1-l_discount)) as sum_disc_price, 
    sum(l_extendedprice*(1-l_discount)*(1+l_tax)) as s ...
params      : {'DELTA': {'type': 'int', 'min': 60, 'max': 120, 'step': 1}}
param_groups: []

=== Q2 ===
select
    s_acctbal,
    s_name,
    n_name,
    p_partkey,
    p_mfgr,
    s_address,
    s_phone,
    s_comment
from
    part,
    supplier,
    partsupp,
    nation,
    region
where
    p_partkey = ps_partkey
    and s_suppkey = ps_sup ...
params      : {'SIZE': {'type': 'int', 'min': 1, 'max': 50, 'step': 1}, 'TYPE': {'type': 'categorical', 'values': ['TIN', 'NICKEL', 'BRASS', 'STEEL', 'COPPER']}, 'REGION': {'type': 'categorical', 'values': ['AFRICA', 'AMERICA', 'ASIA', 'EUROPE', 'MIDDLE EAST']}}
param_groups: []

=== Q3 ===
select l_orderkey,  
    sum(l_extendedprice*(1-l_discount)) as revenue, 
    o_orderdate,  
    o_ship

### Draw parameter instantiations

`query_gen_factory` fills the templates by sampling each placeholder's spec. We draw with a
fixed seed so the instantiations are **identical** for the DuckDB and SynnoDB runs.

In [13]:
N_REPS = 10
rng = random.Random(42)
gen = spec.query_gen_factory(None)

# gen(query_name, rng) -> (query_name, sql_with_values, params_dict)
instantiations = {}
for qid in spec.all_query_ids:
    instantiations[qid] = [gen(f"Q{qid}", rng) for _ in range(N_REPS)]

print(f"Drew {N_REPS} instantiations per query.")
for qid, insts in instantiations.items():
    sample_params = [i[2] for i in insts[:2]]
    print(f"  Q{qid}: {sample_params} ...")

Drew 10 instantiations per query.
  Q1: [{'DELTA': '100'}, {'DELTA': '67'}] ...
  Q2: [{'SIZE': '44', 'TYPE': 'COPPER', 'REGION': 'AFRICA'}, {'SIZE': '38', 'TYPE': 'STEEL', 'REGION': 'AFRICA'}] ...
  Q3: [{'SEGMENT': 'FURNITURE', 'DATE': '1995-03-04'}, {'SEGMENT': 'AUTOMOBILE', 'DATE': '1995-03-13'}] ...
  Q4: [{'DATE': '1997-08-26'}, {'DATE': '1996-07-11'}] ...
  Q5: [{'REGION': 'AMERICA', 'DATE': '1994-08-16'}, {'REGION': 'AFRICA', 'DATE': '1994-04-22'}] ...
  Q6: [{'DATE': '1994-05-17', 'DISCOUNT': '0.04', 'QUANTITY': '25'}, {'DATE': '1995-02-17', 'DISCOUNT': '0.06', 'QUANTITY': '24'}] ...
  Q7: [{'NATION1': 'CHINA', 'NATION2': 'JAPAN'}, {'NATION1': 'IRAQ', 'NATION2': 'GERMANY'}] ...
  Q8: [{'NATION': 'KENYA', 'REGION': 'MIDDLE EAST', 'TYPE': 'MEDIUM PLATED COPPER'}, {'NATION': 'PERU', 'REGION': 'AFRICA', 'TYPE': 'SMALL ANODIZED COPPER'}] ...
  Q9: [{'COLOR': 'puff'}, {'COLOR': 'almond'}] ...
  Q10: [{'DATE': '1993-04-01'}, {'DATE': '1993-10-05'}] ...
  Q11: [{'NATION': 'VIETNAM', '

---
## Step 2 - Generate the SynnoDB Engine

You hand SynnoDB the same `queries.json` and a scale factor. It:

1. **Creates a storage plan** - decides how each query's columns are laid out in memory.
2. **Implements the engine** - writes single-threaded C++, compiles it, validates every output
   against DuckDB, then **auto-publishes** the binary into `ENGINES_DIR`.

This is a one-time cost. Once published the engine is discovered automatically across sessions.

### Start the SynnoDB engine

Constructing the `SynnoDB` driver spawns an in-process **live-UI dashboard** and prints its
URL (e.g. `http://localhost:8765`). Open it in a browser to watch generation unfold in real
time - input tokens, code size, per-query speedups, cost/runtime, and an activity log, all
refreshing every few seconds.

Every stage you chain on this same `db` - storage plan → base implementation →
`runOptimLoop` → `addMultiThreading` → `checkSfCorrectness` - streams onto **one continuous
timeline**, so the whole journey shows up on a single page instead of resetting per stage.
The dashboard stays up for the lifetime of this kernel; the URL is also available as
`db.dashboard_url`.


In [14]:
from synnodb import SynnoDB

db = SynnoDB(
    workload="tpch_byo",
    model=MODEL,
    model_extra_body=MODEL_EXTRA_BODY,
    db_storage="in_memory",
    queries="1-5",
    data_dir=DATA_ROOT,
)

📊 SynnoDB live dashboard: http://localhost:8766


### Storage plan

In [15]:
plan = db.createStoragePlan()

print(plan.text[:600], "...")

2026-07-03 10:58:31 INFO:synnodb.main:Using database source: DBStorage.IN_MEMORY. Disk DB dir: None
2026-07-03 10:58:31 INFO:synnodb.main:RAM preflight passed: RAM sufficient for sf5: dataset 1.59 GiB on disk, 4.77 GiB required in memory (3x), 738.01 GiB available
2026-07-03 10:58:31 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Writing 0 read-only artifact files to `/home/jwehrstein/SynnoDB/tutorials/output` for benchmark tpch_byo
2026-07-03 10:58:31 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Writing 1 artifact files to `/home/jwehrstein/SynnoDB/tutorials/output` for benchmark tpch_byo
2026-07-03 10:58:31 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Writing 1 artifact files (queries.md) to `/home/jwehrstein/SynnoDB/tutorials/output` for optim
2026-07-03 10:58:31 INFO:synnodb.main:Workspace root: output
2026-07-03 10:58:31 INFO:synnodb.utils.model_setup:Using LiteLLM model: anthropic/claude-sonnet-5 (provider: anthropic)
2026-07-03 10:58:31 WARNING:synnod

2026-07-03 10:58:31 INFO:synnodb.conversations.conversation_engine:==================== Prompt ====================
2026-07-03 10:58:31 INFO:synnodb.conversations.conversation_engine:Your task is to analyze the workload and produce a creative in-memory storage-layout summary for the tables accessed by the query. You have the flexibility to return detailed, free-form text that explores not only conventional storage-layout recommendations but also unconventional, novel, and even 'crazy' storage designs.
You are encouraged to include additional ideas, new partitioning strategies, speculative encoding techniques, or experimental ways of grouping and organizing columns or data.
For each accessed table, feel free to be inventive and elaborate on possibilities such as hybrid layouts, speculative SoA/AoS (Array of Structures/Structure of Arrays) approaches, novel column encodings, or adaptive partitioning.
Use this as an opportunity to push beyond current norms and propose storage techniques t

### Base implementation

We feed the plan **content** straight in via `storage_plan=plan.text`, so this step needs
no W&B. If you instead chain off a logged storage-plan run, pass its run id with
`db.createBaseImpl(storage_plan_wandb_id=plan.run_id)`. Provide exactly one of the two.

In [16]:
impl = db.createBaseImpl(storage_plan=plan.text)

print("Workspace :", impl.workspace)
print("Files     :", sorted(impl.files))
print()
print(f"Engine published to: {ENGINES_DIR}")

2026-07-03 11:01:55 INFO:synnodb.main:Using database source: DBStorage.IN_MEMORY. Disk DB dir: None
2026-07-03 11:01:55 INFO:synnodb.main:RAM preflight passed: RAM sufficient for sf5: dataset 1.59 GiB on disk, 4.77 GiB required in memory (3x), 738.08 GiB available
2026-07-03 11:01:55 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Writing 7 read-only artifact files to `/home/jwehrstein/SynnoDB/tutorials/output` for benchmark tpch_byo
2026-07-03 11:01:55 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Writing 7 artifact files (thread_pool.hpp, parquet_reader.cpp, args_parser.hpp, query_impl.hpp, parquet_reader.hpp, query_impl.cpp, query_pool.hpp) to `/home/jwehrstein/SynnoDB/tutorials/output` for optim
2026-07-03 11:01:55 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Writing 16 artifact files to `/home/jwehrstein/SynnoDB/tutorials/output` for benchmark tpch_byo
2026-07-03 11:01:55 INFO:synnodb.cpp_runner.prepare_repo.prepare_workspace:Writing 16 artifact files (db

Registering DuckDB parquet views for SF1: 100%|██████████| 8/8 [00:00<00:00, 2295.10it/s]


2026-07-03 11:31:18 INFO:synnodb.tools.validate.run_and_check_queries:Q1 (OLAPExecSettings(scale_factor=1, db_storage=<DBStorage.IN_MEMORY: 'in_memory'>, parquet_dir=PosixPath('/home/jwehrstein/SynnoDB/.synno_data/workloads/tpch/tpch_parquet/sf1'), disk_db_dir=None)): 65.0ms (Bespoke), 333.20ms (DuckDB)
2026-07-03 11:31:18 INFO:synnodb.tools.validate.run_and_check_queries:Q1 (OLAPExecSettings(scale_factor=1, db_storage=<DBStorage.IN_MEMORY: 'in_memory'>, parquet_dir=PosixPath('/home/jwehrstein/SynnoDB/.synno_data/workloads/tpch/tpch_parquet/sf1'), disk_db_dir=None)): 36.0ms (Bespoke), 316.99ms (DuckDB)
2026-07-03 11:31:18 INFO:synnodb.tools.validate.run_and_check_queries:Q1 (OLAPExecSettings(scale_factor=1, db_storage=<DBStorage.IN_MEMORY: 'in_memory'>, parquet_dir=PosixPath('/home/jwehrstein/SynnoDB/.synno_data/workloads/tpch/tpch_parquet/sf1'), disk_db_dir=None)): 36.0ms (Bespoke), 317.77ms (DuckDB)
2026-07-03 11:31:18 INFO:synnodb.tools.validate.run_and_check_queries:Q1 (OLAPExecSet

Registering DuckDB parquet views for SF2: 100%|██████████| 8/8 [00:00<00:00, 1301.37it/s]


2026-07-03 11:31:27 INFO:synnodb.tools.validate.run_and_check_queries:Q1 (OLAPExecSettings(scale_factor=2, db_storage=<DBStorage.IN_MEMORY: 'in_memory'>, parquet_dir=PosixPath('/home/jwehrstein/SynnoDB/.synno_data/workloads/tpch/tpch_parquet/sf2'), disk_db_dir=None)): 98.0ms (Bespoke), 648.00ms (DuckDB)
2026-07-03 11:31:27 INFO:synnodb.tools.validate.run_and_check_queries:Q1 (OLAPExecSettings(scale_factor=2, db_storage=<DBStorage.IN_MEMORY: 'in_memory'>, parquet_dir=PosixPath('/home/jwehrstein/SynnoDB/.synno_data/workloads/tpch/tpch_parquet/sf2'), disk_db_dir=None)): 73.0ms (Bespoke), 626.25ms (DuckDB)
2026-07-03 11:31:27 INFO:synnodb.tools.validate.run_and_check_queries:Q1 (OLAPExecSettings(scale_factor=2, db_storage=<DBStorage.IN_MEMORY: 'in_memory'>, parquet_dir=PosixPath('/home/jwehrstein/SynnoDB/.synno_data/workloads/tpch/tpch_parquet/sf2'), disk_db_dir=None)): 72.0ms (Bespoke), 629.47ms (DuckDB)
2026-07-03 11:31:27 INFO:synnodb.tools.validate.run_and_check_queries:Q1 (OLAPExecSet

Registering DuckDB parquet views for SF5: 100%|██████████| 8/8 [00:00<00:00, 826.91it/s]


2026-07-03 11:31:51 INFO:synnodb.tools.validate.run_and_check_queries:Q1 (OLAPExecSettings(scale_factor=5, db_storage=<DBStorage.IN_MEMORY: 'in_memory'>, parquet_dir=PosixPath('/home/jwehrstein/SynnoDB/.synno_data/workloads/tpch/tpch_parquet/sf5'), disk_db_dir=None)): 268.0ms (Bespoke), 1628.14ms (DuckDB)
2026-07-03 11:31:51 INFO:synnodb.tools.validate.run_and_check_queries:Q1 (OLAPExecSettings(scale_factor=5, db_storage=<DBStorage.IN_MEMORY: 'in_memory'>, parquet_dir=PosixPath('/home/jwehrstein/SynnoDB/.synno_data/workloads/tpch/tpch_parquet/sf5'), disk_db_dir=None)): 181.0ms (Bespoke), 1571.47ms (DuckDB)
2026-07-03 11:31:51 INFO:synnodb.tools.validate.run_and_check_queries:Q1 (OLAPExecSettings(scale_factor=5, db_storage=<DBStorage.IN_MEMORY: 'in_memory'>, parquet_dir=PosixPath('/home/jwehrstein/SynnoDB/.synno_data/workloads/tpch/tpch_parquet/sf5'), disk_db_dir=None)): 180.0ms (Bespoke), 1592.07ms (DuckDB)
2026-07-03 11:31:51 INFO:synnodb.tools.validate.run_and_check_queries:Q1 (OLAPE

# Step 3a - Benchmark DuckDB
### Run queries on DuckDB as comparison baseline

In [ ]:
import duckdb
from tqdm import tqdm

# Threads both engines run at, so DuckDB and SynnoDB are compared at the same parallelism.
NUM_THREADS = os.cpu_count()  # 1 for demo, else os.cpu_count() for benchmark
assert NUM_THREADS is not None, "os.cpu_count() returned None"

duck = duckdb.connect(":memory:", config={"threads": NUM_THREADS})
duck.execute("PRAGMA disable_progress_bar")
duck.execute("PRAGMA enable_profiling='json'")  # EXPLAIN ANALYZE emits its profile as JSON
for t in TABLES:
    duck.execute(
        f"CREATE VIEW {t} AS SELECT * FROM read_parquet('{SF_DIR}/{t}.parquet')"
    )


def analyze_ms(con, sql):
    """DuckDB's own execution latency for `sql`, via EXPLAIN ANALYZE - the server-side query
    time, excluding the Python call and result-fetch overhead a wall-clock timer would include.
    The json profile (whose `latency` is in seconds) is the second column of the result row."""
    profile = json.loads(con.execute("EXPLAIN ANALYZE " + sql).fetchone()[1])
    return profile["latency"] * 1_000


baseline_times = {}
for qid, insts in tqdm(instantiations.items(), desc="Measuring DuckDB performance"):
    baseline_times[qid] = [analyze_ms(duck, sql) for _, sql, _ in insts]

duck.close()

total_duck = sum(sum(v) / len(v) for v in baseline_times.values())
print(f"{'Query':<8} {'Avg (ms)':>12} {'Median (ms)':>14}")
print("-" * 38)
for qid in spec.all_query_ids:
    t = baseline_times[qid]
    print(f"Q{qid:<7} {sum(t) / len(t):>12.1f} {statistics.median(t):>14.1f}")
print("-" * 38)
print(f"{'TOTAL':<8} {total_duck:>12.1f}")

Running baseline queries: 100%|██████████| 22/22 [03:36<00:00,  9.85s/it]

Query        Avg (ms)    Median (ms)
--------------------------------------
Q1             1206.7         1203.8
Q2              212.6          211.2
Q3              827.9          825.4
Q4              522.0          520.7
Q5             1052.1         1049.4
Q6              513.0          513.9
Q7             1067.9         1065.0
Q8             1284.2         1283.5
Q9             1990.0         1976.3
Q10            1113.9         1111.7
Q11             201.6          201.7
Q12             729.0          742.5
Q13            1409.8         1426.7
Q14             819.1          830.9
Q15             616.4          615.8
Q16             188.9          185.3
Q17             914.7          916.0
Q18            1866.7         1859.8
Q19            1001.8          998.5
Q20            1007.0         1005.3
Q21            2854.4         2856.6
Q22             247.2          245.8
--------------------------------------
TOTAL         21646.9


---
## Step 3 - Drop In SynnoDB

The only change is **one import line** and a few extra keyword arguments to `connect()`:

```diff
- import duckdb
+ import synnodb as duckdb
+ from synnodb.router import RouterMode, RouterPolicy

  con = duckdb.connect(
      ":memory:",
+     config={"threads": NUM_THREADS},   # same knob DuckDB got - fixes the engine's parallelism too
+     engines=str(ENGINES_DIR),
+     policy=RouterPolicy(mode=RouterMode.SAMPLED, cross_check_rate=1.0),
  )
```

Every other line - the view setup, the `execute()` calls, `fetchall()` - is identical.

### Open the drop-in connection

In [28]:
import synnodb as duckdb  # <- only change
from synnodb.router import RouterMode, RouterPolicy

con = duckdb.connect(
    ":memory:",
    config={"threads": NUM_THREADS},  # same thread budget as the DuckDB baseline above
    engines=str(ENGINES_DIR),
    policy=RouterPolicy(mode=RouterMode.SAMPLED, cross_check_rate=1.0),
)

for t in TABLES:
    con.duckdb.execute(
        f"CREATE VIEW {t} AS SELECT * FROM read_parquet('{SF_DIR}/{t}.parquet')"
    )

con.refresh_engines()
n = con.router_stats()["registry"]["templates"]
print(f"Discovered {n} engine template(s) under {ENGINES_DIR}")

2026-07-03 19:03:18 INFO:synnodb.discovery:registered bespoke engine eng-97a825abd0f29634 (4 queries) from /home/jwehrstein/SynnoDB/.synno_data/engines/eng-97a825abd0f29634
Discovered 4 engine template(s) under /home/jwehrstein/SynnoDB/.synno_data/engines


### Run the same queries with the same parameter values

We iterate over `instantiations` - the exact SQL strings from Step 1. For each we record the
engine's **own execution latency** (`engine_ms`, as measured by the router), the same
server-side basis the DuckDB baseline uses via `EXPLAIN ANALYZE` - so the two columns compare
like with like rather than one wall-clock against another.

In [ ]:
synno_times = {}
routed_to = {}  # per query: "SynnoDB" if the bespoke engine served it, else "DuckDB"
for qid, insts in instantiations.items():
    times = []
    backends = []
    for _, sql, _ in insts:
        con.execute(sql).fetchall()
        # `engine_ms` is present when the bespoke engine served the query;
        # a query that falls back to DuckDB exposes `duckdb_ms` instead.
        last = con._last
        served_bespoke = "engine_ms" in last
        times.append(last["engine_ms"] if served_bespoke else last["duckdb_ms"])
        backends.append(served_bespoke)
    synno_times[qid] = times
    routed_to[qid] = "SynnoDB" if all(backends) else "DuckDB"

### Speedup

In [ ]:
total_synno = sum(sum(v)/len(v) for v in synno_times.values())

print(f"{'Query':<8} {'Routing':>8} {'DuckDB (ms)':>12} {'SynnoDB (ms)':>14} {'Speedup':>9}")
print("-" * 55)
for qid in spec.all_query_ids:
    d = sum(baseline_times[qid])/len(baseline_times[qid])
    s = sum(synno_times[qid])/len(synno_times[qid])
    speedup = d / s if s > 0 else float("inf")
    # ⚡ marks queries the bespoke SynnoDB engine served; the rest fell back to DuckDB.
    routing = routed_to[qid]
    mark = " ⚡" if routing == "SynnoDB" else ""
    print(f"Q{qid:<7} {routing:>8} {d:>12.1f} {s:>14.1f} {speedup:>8.2f}x{mark}")
print("-" * 55)
overall = total_duck / total_synno if total_synno > 0 else float("inf")
print(f"{'TOTAL':<8} {'':>8} {total_duck:>12.1f} {total_synno:>14.1f} {overall:>8.2f}x")

### Correctness guarantee

Every routed result was compared against DuckDB (`cross_check_rate=1.0`).
The mismatch count must be 0.

In [32]:
stats = con.router_stats()["session"]
print(f"Routed:         {stats['routed']}")
print(f"Cross-checked:  {stats['cross_checked']}")
print(f"Mismatches:     {stats['cross_check_mismatch']}")
assert stats["cross_check_mismatch"] == 0, "result divergence detected!"
print("\nAll results match DuckDB exactly.")

Routed:         40
Cross-checked:  40
Mismatches:     0

All results match DuckDB exactly.


### Q1 result (with timing footer)

In [33]:
_, q1_sql, _ = instantiations["1"][0]
con.execute(q1_sql)
con.show()
con.close()

2026-07-03 19:11:35 INFO:synnodb.router:router: routed eng-97a825abd0f29634::1: bespoke 187.3ms vs duckdb 1234.6ms → 6.6× speedup, results ✓
┌──────────────┬──────────────┬───────────────────┬───────────────────┬───────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬─────────────┐
│ l_returnflag │ l_linestatus │      sum_qty      │   sum_base_price  │   sum_disc_price  │      sum_charge     │      avg_qty       │     avg_price      │       avg_disc      │ count_order │
│    string    │    string    │ decimal128(38, 2) │ decimal128(38, 2) │ decimal128(38, 4) │  decimal128(38, 6)  │       double       │       double       │        double       │    int64    │
├──────────────┼──────────────┼───────────────────┼───────────────────┼───────────────────┼─────────────────────┼────────────────────┼────────────────────┼─────────────────────┼─────────────┤
│ A            │ F            │ 188818373.00      │ 283107483036.12   │ 268952035589.0630 │

---
## Bonus - Define Your Own Conversation

Every built-in stage above is an ordinary `ConversationPlan` - and you can assemble
your own from the same primitives. A plan names the run (for logging and caching),
states *what the prepared workspace must provide* (`PrepareFeatures`), and supplies a
`stages` callable that turns a `ConvContext` into a flat list of stage items:

- `PromptStage` - one declarative LLM task with measurement/revert flags; its prompt
  callback receives the freshly measured runtime (and trace data, if requested).
- `PerQueryLoop` - one conversation branch per query, stages executed ring by ring.
- markers (`Compact`, `Benchmark`, `ValidateOn`, ...) and checks (`AssertCorrect`).

`db.run_synthesis(plan, start=...)` is the single entry point every stage goes
through; `start` chains off any earlier artifact (here: the base implementation).
The returned artifact carries the final snapshot hash and the workspace's prepare
record, so it chains onwards (e.g. into `db.checkSfCorrectness(result, target_sf=100)`).


In [25]:
from synnodb import (
    AssertCorrect, Benchmark, Compact, ConversationPlan, ConvContext,
    PerQueryLoop, PrepareFeatures, PromptStage,
)

def my_stages(ctx: ConvContext):
    return [
        AssertCorrect(),
        PromptStage(
            descriptor="inspect hot loops",
            get_prompt=lambda _exec_settings, _rt: (
                f"Profile {ctx.filenames.query_impl_path} and summarize the hot loops."),
            measure_performance_after_stage=False,
            auto_revert_on_regression=False,
        ),
        Compact(),
        PerQueryLoop(lambda qid, ctx: [
            PromptStage(
                descriptor=f"tune {qid}",
                get_prompt_with_tracing=lambda _exec_settings, rt, trace: (
                    f"Query {qid} currently runs in {rt:.0f} ms.\n"
                    f"Trace:\n{trace}\nOptimize it."),
                max_turns=125,
                # defaults: measure after stage, auto-revert on regression
            ),
        ]),
        Benchmark(),
    ]

tuning_plan = ConversationPlan(
    name="myTuningPass",                    # run identity: naming, logging, caching
    prepare=PrepareFeatures(tracing=True),  # the workspace needs tracing instrumentation
    stages=my_stages,
)

# Uncomment to run the custom pass on top of the base implementation:
# tuned = db.run_synthesis(tuning_plan, start=impl)


---
## Where to go next

The base implementation is single-threaded. The same `SynnoDB` object carries each engine further:

```python
opt   = db.runOptimLoop(base_impl=impl)           # single-threaded SIMD / cache optimization
multi = db.addMultiThreading(optimized=opt)        # parallel execution across cores
rep   = db.checkSfCorrectness(source=multi, target_sf=50)  # correctness at larger SF
```

These chain **without W&B**: each stage restores the previous one's engine straight from
its git snapshot (`impl.snapshot_hash`, `opt.snapshot_hash`, ...) in the local workspace, so
the whole pipeline runs for non-W&B users out of the box. (This works within one session /
workspace; to chain across machines, log to W&B and pass the run id instead, e.g.
`db.runOptimLoop(base_impl_wandb_id=impl.run_id)`.)

> **Want W&B logging?** Pass `wandb_project="..."` (and/or `wandb_entity="..."`) to
> `SynnoDB(...)`. W&B is off unless one of them is set — nothing logs in, initializes, or
> requires credentials otherwise.